# 01 — Pan-Organ Net V2: Data Ingestion & Sharding (100% Google Drive Powered)

Direct ingestion from Google Drive . Zero Kaggle dependency.

In [ ]:
import os, sys, shutil
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive', force_remount=False)

DRIVE_ROOT = Path('/content/drive/MyDrive/ResearchPilot_PanOrgan/V2')
CODE_DIR = DRIVE_ROOT / 'code'
SHARDS_DIR = DRIVE_ROOT / 'preprocessed_shards'
CKPT_DIR = DRIVE_ROOT / 'checkpoints'
EXP_DIR = DRIVE_ROOT / 'experiments'
RAW_DRIVE = DRIVE_ROOT / 'raw_archives'

for d in [SHARDS_DIR, CKPT_DIR, EXP_DIR, RAW_DRIVE]:
    d.mkdir(parents=True, exist_ok=True)

# Unpack code package onto local fast SSD
if (CODE_DIR / 'V2_code_package.tar.gz').exists() and not Path('/content/panorgan_v2').exists():
    Path('/content/panorgan_v2').mkdir(parents=True, exist_ok=True)
    !tar -xzf "{CODE_DIR}/V2_code_package.tar.gz" -C /content/panorgan_v2

PKG_ROOT = Path('/content/panorgan_v2') if Path('/content/panorgan_v2/pyproject.toml').exists() else CODE_DIR
if str(PKG_ROOT) not in sys.path:
    sys.path.insert(0, str(PKG_ROOT))
!pip install -q --no-deps -e "{PKG_ROOT}"
!pip install -q timm nibabel pydicom pyarrow scikit-learn scipy PyYAML tqdm open-clip-torch

import importlib
importlib.invalidate_caches()
import torch
from panorgan.config import load_config, detect_runtime_profile, seed_everything

cfg = load_config(PKG_ROOT / 'configs' / 'v2_panorgan.yaml' if (PKG_ROOT / 'configs' / 'v2_panorgan.yaml').exists() else CODE_DIR / 'v2_panorgan.yaml')
seed_everything(cfg['project']['seed'], cfg['project']['deterministic'])
runtime = detect_runtime_profile(cfg)
print(f'Active Runtime -> GPU: {runtime.gpu_name} | Tier: {runtime.tier} | Batch: {runtime.batch_size}x{runtime.grad_accum} | AMP: {runtime.amp_dtype}')


## Step 1: Verify Google Drive Raw Archives (Zero External Login Needed)
All medical cohorts are pre-uploaded to Google Drive .
No Kaggle API, no Kaggle secrets, and no external tokens are required.


In [ ]:
RAW_DRIVE = DRIVE_ROOT / 'raw_archives'
print('=== Verified Raw Datasets Available in Google Drive ===')
available_archives = {}
for p in sorted(RAW_DRIVE.iterdir()):
    if p.is_file():
        available_archives[p.name] = p
        print(f'  [✓] {p.name:<32} ({p.stat().st_size / (1024*1024):.1f} MB)')

print('[INFO] 100% Google Drive mode active. Zero Kaggle credentials needed!')

## Step 2: Physics Harmonization, Zero-Leakage Splits & 256 MB WebDataset/NPZ Sharding

In [ ]:
import json
import tarfile
import numpy as np
import pandas as pd
import nibabel as nib
import pydicom
from PIL import Image
from tqdm import tqdm

from panorgan.data.harmonizer import (
    build_25d_slab,
    harmonize_projection_2d,
    harmonize_ultrasound_2d,
    harmonize_mri_slice,
    resize_with_aspect_pad,
)
from panorgan.data.datasets_registry import (
    parse_amos22_volumes,
    parse_busi_dataset,
    parse_camus_cardiac_us,
    parse_cbis_ddsm_curated,
    parse_nih_cxr14_dataset,
    parse_rsna_pneumonia_external,
    parse_tcga_lgg_mri,
    parse_ultrasound_nerve_dataset,
    strip_dicom_phi,
)
from panorgan.data.shard_builder import build_patient_splits, verify_zero_patient_overlap, write_dataset_shards

SCRATCH = Path(cfg['storage']['local_scratch_dir'])
SCRATCH.mkdir(parents=True, exist_ok=True)
SHARD_BYTES = int(cfg['storage']['shard_size_mb']) * 1024 * 1024

all_manifest_rows = []


def _load_2d_array(path_str: str) -> np.ndarray:
    p = Path(path_str)
    if p.suffix.lower() == '.dcm':
        dcm = pydicom.dcmread(str(p))
        strip_dicom_phi(dcm)
        return dcm.pixel_array.astype(np.float32)
    if p.name.endswith('.nii.gz') or p.suffix.lower() == '.nii':
        arr = np.asarray(nib.load(str(p)).dataobj, dtype=np.float32)
        return arr[:, :, arr.shape[2] // 2] if arr.ndim == 3 else arr
    with Image.open(p) as im:
        return np.asarray(im.convert('L'), dtype=np.float32)


def _load_mask_array(mask_ref: str | None, target_hw: tuple[int, int]) -> np.ndarray | None:
    if not mask_ref:
        return None
    parts = [s for s in str(mask_ref).split(';') if s and Path(s).exists()]
    if not parts:
        return None
    union = None
    for part in parts:
        p = Path(part)
        if p.name.endswith('.nii.gz') or p.suffix.lower() == '.nii':
            arr = np.asarray(nib.load(str(p)).dataobj)
            if arr.ndim == 3:
                arr = arr[:, :, arr.shape[2] // 2]
            m = arr.astype(np.uint8)
        else:
            with Image.open(p) as im:
                m = (np.asarray(im.convert('L')) > 127).astype(np.uint8)
        union = m if union is None else np.maximum(union, m)
    return resize_with_aspect_pad(union, target_hw=target_hw, is_mask=True, pad_value=0)


def process_2d_records(records, ds_key: str, modality: str):
    if not records:
        print(f'[SKIP] No records parsed for {ds_key}')
        return
    df = pd.DataFrame([r.__dict__ for r in records])
    df = build_patient_splits(df, seed=cfg['project']['seed'])
    target_hw = tuple(cfg['modalities'][modality]['target_size'])

    def _gen():
        for _, row in tqdm(df.iterrows(), total=len(df), desc=f'Sharding {ds_key}'):
            try:
                raw_arr = _load_2d_array(row['image_path'])
            except Exception:
                continue
            if modality == 'US':
                img = harmonize_ultrasound_2d(raw_arr, target_hw=target_hw)
            elif modality == 'MRI':
                img = harmonize_mri_slice(raw_arr, target_hw=target_hw)
            else:
                img = harmonize_projection_2d(raw_arr, target_hw=target_hw)
            mask = _load_mask_array(row.get('mask_path'), target_hw=target_hw)
            yield {
                'sample_id': row['sample_id'],
                'image': img,
                'mask': mask,
                'meta': {
                    'dataset': row['dataset'],
                    'patient_id': row['patient_id'],
                    'modality': row['modality'],
                    'specialty': row.get('specialty', 'general_radiology'),
                    'task': row.get('task', 'classification'),
                    'split': row['split'],
                    'label': None if pd.isna(row.get('label')) else int(row['label']),
                    'multilabel': row.get('multilabel') if isinstance(row.get('multilabel'), list) else None,
                    'spacing_mm': list(row.get('spacing_mm', [1.0, 1.0])),
                    'body_coord_z': float(row.get('body_coord_z', 0.5)),
                    'is_3d': False,
                    'has_contrast': bool(row.get('has_contrast', False)),
                    'prompt': row.get('prompt', ''),
                },
            }

    dest_dir = SHARDS_DIR / ds_key
    dest_dir.mkdir(parents=True, exist_ok=True)
    rows = write_dataset_shards(_gen(), dest_dir, shard_prefix=ds_key, max_shard_bytes=SHARD_BYTES)
    all_manifest_rows.extend(rows)
    print(f'[DONE] {ds_key}: {len(rows)} samples written to Drive shards.')


In [ ]:
# --- 1. Load Clinical Datasets Directly from Google Drive raw_archives ---
DRIVE_DATASETS = [
    ('BUSI_raw.tar.gz', 'busi', 'US', lambda p: parse_busi_dataset(p, deduplicate=True)),
    ('CBIS_DDSM_raw.tar.gz', 'cbis_ddsm', 'MG', lambda p: parse_cbis_ddsm_curated(p)),
    ('TCGA_LGG_raw.zip', 'tcga_lgg', 'MRI', lambda p: parse_tcga_lgg_mri(p)),
    ('Ultrasound_Nerve_raw.tar.gz', 'ultrasound_nerve', 'US', lambda p: parse_ultrasound_nerve_dataset(p)),
    ('NIH_CXR14_raw.tar.gz', 'nih_cxr14', 'CXR', lambda p: parse_nih_cxr14_dataset(p, max_patients=3000)),
]

for arc_name, ds_key, mod, parser_fn in DRIVE_DATASETS:
    arc = RAW_DRIVE / arc_name
    if not arc.exists():
        print(f'[SKIP] {arc_name} not found in {RAW_DRIVE}; skipping.')
        continue
    work = SCRATCH / ds_key
    shutil.rmtree(work, ignore_errors=True)
    work.mkdir(parents=True, exist_ok=True)
    print(f'[INFO] Extracting {arc_name} from Drive to fast local NVMe SSD...')
    try:
        if arc.name.endswith('.tar.gz') or arc.name.endswith('.tgz'):
            !tar -xzf "{arc}" -C "{work}"
        elif arc.name.endswith('.zip'):
            !unzip -qo "{arc}" -d "{work}"
        recs = parser_fn(work)
        print(f'Parsed {len(recs)} clinical samples for {ds_key}. Harmonizing and sharding...')
        process_2d_records(recs, ds_key=ds_key, modality=mod)
    except Exception as exc:
        print(f'[WARN] {ds_key} encountered an issue: {exc}')
    finally:
        shutil.rmtree(work, ignore_errors=True)

In [ ]:
# --- 2. AMOS22 3D Multi-Organ CT + MRI -> 2.5D Volumetric Slabs ---
amos_arc = RAW_DRIVE / 'amos22.zip'
amos_work = SCRATCH / 'amos22'
shutil.rmtree(amos_work, ignore_errors=True)
amos_work.mkdir(parents=True, exist_ok=True)
amos_zip = amos_work / 'amos22.zip'

if amos_arc.exists():
    print('[INFO] Found amos22.zip in Google Drive! Unpacking...')
    !unzip -qo "{amos_arc}" -d "{amos_work}"
else:
    print('[INFO] Downloading open-access AMOS22 from Zenodo...')
    !curl -L -C - "https://zenodo.org/records/7155725/files/amos22.zip?download=1" -o "{amos_zip}"
    !unzip -qo "{amos_zip}" -d "{amos_work}" && rm -f "{amos_zip}"

amos_root = next((p for p in [amos_work / 'amos22', amos_work] if (p / 'imagesTr').exists()), amos_work)
ct_recs, mri_recs = parse_amos22_volumes(amos_root, slice_stride=3, min_foreground_pixels=128)
for recs, ds_key, mod in [(ct_recs, 'amos22_ct', 'CT'), (mri_recs, 'amos22_mri', 'MRI')]:
    if not recs:
        continue
    df = build_patient_splits(pd.DataFrame([r.__dict__ for r in recs]), seed=cfg['project']['seed'])
    target_hw = tuple(cfg['modalities'][mod]['target_size'])
    slab_d = int(cfg['modalities'][mod]['slab_depth'])

    def _gen_vol():
        for img_path, grp in tqdm(df.groupby('image_path'), desc=f'Sharding {ds_key}'):
            vol = np.asarray(nib.load(img_path).dataobj, dtype=np.float32).transpose(2, 0, 1)
            lbl_vol = np.asarray(nib.load(grp.iloc[0]['mask_path']).dataobj, dtype=np.uint8).transpose(2, 0, 1)
            for _, row in grp.iterrows():
                z = int(row['slice_index'])
                slab = build_25d_slab(vol, center_idx=z, slab_depth=slab_d, modality=mod, target_hw=target_hw)
                msk = resize_with_aspect_pad(lbl_vol[z], target_hw=target_hw, is_mask=True, pad_value=0)
                yield {
                    'sample_id': row['sample_id'],
                    'image': slab,
                    'mask': msk,
                    'meta': {
                        'dataset': row['dataset'],
                        'patient_id': row['patient_id'],
                        'modality': mod,
                        'specialty': 'abdominal_radiology',
                        'task': 'segmentation',
                        'split': row['split'],
                        'label': None,
                        'multilabel': None,
                        'spacing_mm': list(row['spacing_mm']),
                        'body_coord_z': float(row['body_coord_z']),
                        'is_3d': True,
                        'has_contrast': bool(row['has_contrast']),
                        'prompt': row['prompt'],
                    },
                }

    dest_dir = SHARDS_DIR / ds_key
    dest_dir.mkdir(parents=True, exist_ok=True)
    rows = write_dataset_shards(_gen_vol(), dest_dir, shard_prefix=ds_key, max_shard_bytes=SHARD_BYTES)
    all_manifest_rows.extend(rows)
    print(f'[DONE] {ds_key}: {len(rows)} volumetric slabs written to Drive.')

shutil.rmtree(amos_work, ignore_errors=True)


## Step 3: Enforce Zero-Patient-Leakage Audit Certificate & Save Master Manifest

In [ ]:
manifest_df = pd.DataFrame(all_manifest_rows)
audit_report = verify_zero_patient_overlap(manifest_df)

manifest_path = SHARDS_DIR / 'master_manifest.parquet'
audit_path = SHARDS_DIR / 'split_audit_report.json'
manifest_df.to_parquet(manifest_path, index=False)
audit_path.write_text(json.dumps(audit_report, indent=2), encoding='utf-8')

print('=== ZERO-LEAKAGE AUDIT PASSED ===')
print(json.dumps(audit_report, indent=2))
!df -h /content
